# KURE 번아웃 합성 데이터 + E2E 재학습 v2

**목적**: v1 실험(1:3 / 1:1 / 3:1) 결과 분석 후, 고비율 범위(5:1 / 7:1) 추가 탐색

## 배경
- v1 결과: 합성 비율 증가 → 단조 성능 향상 (1:3 < 1:1 < 3:1, F1 0.4835)
- 포화점 미도달 가능성 → 5:1 / 7:1 추가 실험

## 실험 계획
| 실험 | 합성:원본 비율 | 예상 Train 크기 |
|------|--------------|----------------|
| ratio_5to1 | 5:1 | 원본 + 5배 합성 (max 가용) |
| ratio_7to1 | 7:1 | 원본 + 7배 합성 (max 가용) |

## 파이프라인
```
stage2_train_v3.csv (원본)
diary_synthetic.csv (합성, Ollama 생성)
        ↓ 비율별 혼합
KURE 백본 (상위 2레이어 학습) → mean pooling → MLP 분류기
        ↓ warm-start from stage2_model_v3.pt
2개 비율 실험 → 최고 F1 모델 저장
```


## 1. 환경 설정

In [2]:
!nvidia-smi
!pip install -q transformers accelerate sentence-transformers scikit-learn
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124

Thu Mar 26 11:38:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.59                 Driver Version: 591.59         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   55C    P5              5W /   83W |    2226MiB /   8188MiB |     16%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import os, warnings, copy
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader, TensorDataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm
warnings.filterwarnings('ignore')
import torch
print(torch.__version__)  # 2.6.0+cu124 나와야 함
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

D:\Programming\Projects\Burnout\llm\.venv1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.6.0+cu124
Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.6 GB


## 2. 경로 설정

In [4]:

# ── Colab (Drive) ──────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH  = '/content/drive/MyDrive/Burnout'
# MODEL_PATH = '/content/drive/MyDrive/Burnout/models'
# CKPT_PATH  = '/content/drive/MyDrive/Burnout/checkpoints'

# ── 로컬 ────────────────────────────────────────────
DATA_PATH  = 'D:/Programming/Projects/Burnout/llm/dataset'
MODEL_PATH = 'D:/Programming/Projects/Burnout/llm/models'
CKPT_PATH  = 'D:/Programming/Projects/Burnout/llm/checkpoints'

STAGE2_CATEGORIES = {0: '정서적_고갈', 1: '좌절_압박', 2: '부정적_대인관계', 3: '자기비하'}
CAT_TO_IDX = {v: k for k, v in STAGE2_CATEGORIES.items()}

# 입력
S2_TRAIN_PATH  = f'{DATA_PATH}/stage2_train_v3.csv'
S2_VAL_PATH    = f'{DATA_PATH}/stage2_val_v3.csv'
SYNTHETIC_PATH = f'{DATA_PATH}/diary_synthetic.csv'
WARMSTART_PATH = f'{MODEL_PATH}/stage2_model_v3.pt'

# 출력 — 최종 모델
SAVE_PATHS = {
    'ratio_5to1': f'{MODEL_PATH}/stage2_model_syn_5to1.pt',
    'ratio_7to1': f'{MODEL_PATH}/stage2_model_syn_7to1.pt',
}

# 출력 — 에폭 체크포인트
EPOCH_CKPT_PATHS = {
    'ratio_5to1': f'{CKPT_PATH}/ckpt_epoch_syn_5to1.pt',
    'ratio_7to1': f'{CKPT_PATH}/ckpt_epoch_syn_7to1.pt',
}

os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(CKPT_PATH, exist_ok=True)

print('경로 확인:')
for name, path in [('S2 Train', S2_TRAIN_PATH), ('S2 Val', S2_VAL_PATH),
                   ('Synthetic', SYNTHETIC_PATH), ('Warm-start', WARMSTART_PATH)]:
    print(f'  {"✅" if os.path.exists(path) else "❌"} {name}: {path}')


경로 확인:
  ✅ S2 Train: D:/Programming/Projects/Burnout/llm/dataset/stage2_train_v3.csv
  ✅ S2 Val: D:/Programming/Projects/Burnout/llm/dataset/stage2_val_v3.csv
  ✅ Synthetic: D:/Programming/Projects/Burnout/llm/dataset/diary_synthetic.csv
  ✅ Warm-start: D:/Programming/Projects/Burnout/llm/models/stage2_model_v3.pt


## 3. 데이터 로드

In [5]:
# 원본 데이터
s2_train_orig = pd.read_csv(S2_TRAIN_PATH)
s2_val        = pd.read_csv(S2_VAL_PATH)

print(f'원본 Train: {len(s2_train_orig):,}건')
print(f'Val       : {len(s2_val):,}건')
print()
print('[원본 Train 클래스 분포]')
for label, cat in STAGE2_CATEGORIES.items():
    cnt = (s2_train_orig['label'] == label).sum()
    print(f'  {cat}: {cnt:,}건 ({cnt/len(s2_train_orig)*100:.1f}%)')

원본 Train: 39,547건
Val       : 4,395건

[원본 Train 클래스 분포]
  정서적_고갈: 10,498건 (26.5%)
  좌절_압박: 9,969건 (25.2%)
  부정적_대인관계: 9,634건 (24.4%)
  자기비하: 9,446건 (23.9%)


In [6]:
# 합성 데이터
s2_synthetic = pd.read_csv(SYNTHETIC_PATH)

# category 컬럼 → label 인덱스 변환
s2_synthetic['label'] = s2_synthetic['category'].map(CAT_TO_IDX)
s2_synthetic = s2_synthetic[['text', 'label']].dropna()
s2_synthetic['label'] = s2_synthetic['label'].astype(int)

print(f'합성 데이터: {len(s2_synthetic):,}건')
print()
print('[합성 데이터 클래스 분포]')
for label, cat in STAGE2_CATEGORIES.items():
    cnt = (s2_synthetic['label'] == label).sum()
    print(f'  {cat}: {cnt:,}건 ({cnt/len(s2_synthetic)*100:.1f}%)')

합성 데이터: 4,026건

[합성 데이터 클래스 분포]
  정서적_고갈: 1,000건 (24.8%)
  좌절_압박: 1,026건 (25.5%)
  부정적_대인관계: 1,000건 (24.8%)
  자기비하: 1,000건 (24.8%)


## 4. 데이터 혼합 유틸리티

- 비율은 **합성:원본** 기준
- 카테고리별로 비율 적용 → 클래스 균형 유지
- 합성 데이터가 부족하면 가용 최대치만 사용

In [7]:
def mix_datasets(orig, synthetic, syn_ratio, orig_ratio=1.0, random_state=42):
    """
    원본과 합성 데이터를 카테고리별로 비율에 맞게 혼합.
    take_syn = min(목표량, 가용 최대치) 이므로 합성 데이터 부족 시 경고 출력.
    """
    pieces = []
    for label in STAGE2_CATEGORIES:
        orig_cat = orig[orig['label'] == label]
        syn_cat  = synthetic[synthetic['label'] == label]

        pieces.append(orig_cat)

        target_syn = int(len(orig_cat) * syn_ratio / orig_ratio)
        take_syn   = min(target_syn, len(syn_cat))

        if take_syn > 0:
            sampled = syn_cat.sample(n=take_syn, random_state=random_state)
            pieces.append(sampled)

    mixed = pd.concat(pieces, ignore_index=True).sample(
        frac=1, random_state=random_state
    ).reset_index(drop=True)
    return mixed


# 실험할 비율 정의
RATIOS = {
    'ratio_5to1': (5, 1),   # 합성 : 원본 = 5 : 1
    'ratio_7to1': (7, 1),   # 합성 : 원본 = 7 : 1
}

print('[혼합 데이터 크기 미리보기]')
for ratio_name, (syn_r, orig_r) in RATIOS.items():
    mixed = mix_datasets(s2_train_orig, s2_synthetic, syn_ratio=syn_r, orig_ratio=orig_r)
    orig_cnt = len(s2_train_orig)
    syn_cnt  = len(mixed) - orig_cnt
    print(f'  {ratio_name} ({syn_r}:{orig_r}): 총 {len(mixed):,}건 (원본 {orig_cnt:,} + 합성 {syn_cnt:,})')
    ideal = int(orig_cnt * syn_r / orig_r)
    if syn_cnt < ideal:
        print(f'    ⚠️  합성 데이터 부족: 목표 {ideal:,}건, 실제 {syn_cnt:,}건 (가용 최대치 사용)')


[혼합 데이터 크기 미리보기]
  ratio_5to1 (5:1): 총 43,573건 (원본 39,547 + 합성 4,026)
    ⚠️  합성 데이터 부족: 목표 197,735건, 실제 4,026건 (가용 최대치 사용)
  ratio_7to1 (7:1): 총 43,573건 (원본 39,547 + 합성 4,026)
    ⚠️  합성 데이터 부족: 목표 276,829건, 실제 4,026건 (가용 최대치 사용)


## 5. 모델 정의 (E2E v1 동일 구조)

In [8]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.05):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.weight,
                             label_smoothing=self.label_smoothing, reduction='none')
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()


class E2EBurnoutModel(nn.Module):
    def __init__(self, backbone, hidden_dim=256, num_classes=4, dropout=0.2):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Linear(1024, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def mean_pool(self, token_embeds, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        return (token_embeds * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.mean_pool(out.last_hidden_state, attention_mask)
        return self.classifier(pooled)

In [9]:
# KURE 로드 (토크나이저 + 백본)
print('KURE 로딩 중...')
st_model   = SentenceTransformer('nlpai-lab/KURE-v1')
tokenizer  = st_model.tokenizer
backbone_orig = st_model[0].auto_model   # 원본 백본 보관 (매 실험마다 복사해서 사용)
del st_model
torch.cuda.empty_cache()
print('✅ KURE 로드 완료')

MAX_LEN    = 128
NUM_UNFREEZE = 2  # E2E v1 동일


def tokenize(texts):
    return tokenizer(
        texts, padding='max_length', truncation=True,
        max_length=MAX_LEN, return_tensors='pt'
    )


def build_model(warmstart_path: str) -> E2EBurnoutModel:
    """매 실험마다 새 모델 인스턴스 생성 (백본 deep copy + 레이어 해제)"""
    backbone = copy.deepcopy(backbone_orig)

    # 전체 고정
    for param in backbone.parameters():
        param.requires_grad = False

    # 상위 2레이어 해제
    layers = None
    if hasattr(backbone, 'encoder') and hasattr(backbone.encoder, 'layer'):
        layers = backbone.encoder.layer
    elif hasattr(backbone, 'roberta'):
        layers = backbone.roberta.encoder.layer

    if layers is not None:
        for layer in layers[-NUM_UNFREEZE:]:
            for param in layer.parameters():
                param.requires_grad = True

    if hasattr(backbone, 'gradient_checkpointing_enable'):
        backbone.gradient_checkpointing_enable()

    model = E2EBurnoutModel(backbone, hidden_dim=256, num_classes=4, dropout=0.2).to(device)

    # 분류기 warm-start
    if os.path.exists(warmstart_path):
        ckpt = torch.load(warmstart_path, map_location='cpu', weights_only=False)
        cls_state = {k.replace('classifier.', ''): v
                     for k, v in ckpt['model_state_dict'].items()}
        model.classifier.load_state_dict(cls_state)
        print(f'  ✅ 분류기 warm-start: {os.path.basename(warmstart_path)}')
    else:
        print('  ⚠️ warm-start 모델 없음 → random init')

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'  파라미터: {trainable/1e6:.1f}M 학습 / {total/1e6:.1f}M 전체')
    return model


# Val 토크나이징 (공통)
print('\nVal 토크나이징...')
val_enc    = tokenize(s2_val['text'].tolist())
val_labels = torch.tensor(s2_val['label'].values, dtype=torch.long)
val_ds     = TensorDataset(val_enc['input_ids'], val_enc['attention_mask'], val_labels)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
print('✅ 완료')

KURE 로딩 중...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 754.72it/s, Materializing param=pooler.dense.weight]                               


✅ KURE 로드 완료

Val 토크나이징...
✅ 완료


## 6. 학습 함수

In [10]:
E2E_CONFIG = {
    'epochs': 30,
    'batch_size': 16,
    'lr_backbone': 1e-5,
    'lr_head': 1e-4,
    'weight_decay': 1e-4,
    'patience': 5,
    'warmup_epochs': 2,
    'focal_gamma': 2.0,
    'label_smoothing': 0.05,
}


def train_one_ratio(ratio_name: str, train_df: pd.DataFrame,
                    save_path: str, data_version: str,
                    epoch_ckpt_path: str) -> dict:
    """단일 비율 실험 학습 → best F1 반환. 에폭 체크포인트로 중단/재개 지원."""
    print(f'\n{"="*60}')
    print(f'실험: {ratio_name}  |  Train {len(train_df):,}건')
    print(f'{"="*60}')

    # 데이터셋 준비
    print('토크나이징 중...')
    train_enc    = tokenize(train_df['text'].tolist())
    train_labels = torch.tensor(train_df['label'].values, dtype=torch.long)
    train_ds     = TensorDataset(train_enc['input_ids'], train_enc['attention_mask'], train_labels)
    train_loader = DataLoader(train_ds, batch_size=E2E_CONFIG['batch_size'], shuffle=True,
                              generator=torch.Generator().manual_seed(42))
    print(f'  배치 수: {len(train_loader)}')

    # 모델 + 옵티마이저 초기화
    print('모델 초기화...')
    model = build_model(WARMSTART_PATH)

    counts = torch.bincount(train_labels, minlength=4).float()
    class_weights = (1.0 / counts.clamp(min=1)).to(device)
    class_weights = class_weights / class_weights.sum() * 4
    criterion = FocalLoss(gamma=E2E_CONFIG['focal_gamma'], weight=class_weights,
                          label_smoothing=E2E_CONFIG['label_smoothing'])

    backbone_params = [p for p in model.backbone.parameters() if p.requires_grad]
    head_params     = list(model.classifier.parameters())
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': E2E_CONFIG['lr_backbone'],
         'weight_decay': E2E_CONFIG['weight_decay']},
        {'params': head_params, 'lr': E2E_CONFIG['lr_head'],
         'weight_decay': E2E_CONFIG['weight_decay']},
    ])
    scheduler    = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=E2E_CONFIG['epochs'])
    scaler       = GradScaler()

    best         = {'f1': 0.0, 'acc': 0.0, 'epoch': 0}
    patience_cnt = 0
    start_epoch  = 0

    # ── 에폭 체크포인트 로드 ──────────────────────────────────
    if os.path.exists(epoch_ckpt_path):
        print(f'  에폭 체크포인트 로드 중: {os.path.basename(epoch_ckpt_path)}')
        ep_ckpt = torch.load(epoch_ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ep_ckpt['model_state_dict'])
        optimizer.load_state_dict(ep_ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ep_ckpt['scheduler_state_dict'])
        scaler.load_state_dict(ep_ckpt['scaler_state_dict'])
        best         = ep_ckpt['best']
        patience_cnt = ep_ckpt['patience_cnt']
        start_epoch  = ep_ckpt['epoch']
        print(f'  ✅ {start_epoch}에폭부터 재개 (현재 best F1 {best["f1"]:.4f})')
    # ─────────────────────────────────────────────────────────

    print(f'학습 시작 (에폭 {start_epoch+1} ~ {E2E_CONFIG["epochs"]})')
    for epoch in range(start_epoch, E2E_CONFIG['epochs']):
        # Warmup
        if epoch < E2E_CONFIG['warmup_epochs']:
            scale = (epoch + 1) / E2E_CONFIG['warmup_epochs']
            optimizer.param_groups[0]['lr'] = E2E_CONFIG['lr_backbone'] * scale
            optimizer.param_groups[1]['lr'] = E2E_CONFIG['lr_head'] * scale

        # Train
        model.train()
        train_loss = 0.0
        for input_ids, attention_mask, lbl in train_loader:
            input_ids      = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            lbl            = lbl.to(device)
            optimizer.zero_grad()
            with autocast():
                logits = model(input_ids, attention_mask)
                loss   = criterion(logits, lbl)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        if epoch >= E2E_CONFIG['warmup_epochs']:
            scheduler.step()

        # Validate
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for input_ids, attention_mask, lbl in val_loader:
                with autocast():
                    logits = model(input_ids.to(device), attention_mask.to(device))
                all_preds.extend(logits.argmax(dim=1).cpu().tolist())
                all_labels.extend(lbl.tolist())

        val_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
        val_f1  = f1_score(all_labels, all_preds, average='macro')

        if val_f1 > best['f1']:
            best = {'f1': val_f1, 'acc': val_acc, 'epoch': epoch + 1}
            torch.save({
                'model_state_dict': model.state_dict(),
                'num_unfreeze_layers': NUM_UNFREEZE,
                'num_classes': 4,
                'categories': STAGE2_CATEGORIES,
                'config': E2E_CONFIG,
                'best_metrics': best,
                'data_version': data_version,
                'ratio_name': ratio_name,
            }, save_path)
            patience_cnt = 0
            marker = ' ★'
        else:
            patience_cnt += 1
            marker = ''

        if (epoch + 1) % 5 == 0 or marker:
            print(f'  Epoch {epoch+1:3d} | Loss {train_loss:.4f} | '
                  f'Acc {val_acc:.4f} | F1 {val_f1:.4f}{marker}')

        # ── 에폭 체크포인트 저장 (매 에폭) ──────────────────────
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best': best,
            'patience_cnt': patience_cnt,
        }, epoch_ckpt_path)
        # ─────────────────────────────────────────────────────────

        if patience_cnt >= E2E_CONFIG['patience']:
            print(f'  Early stopping at epoch {epoch + 1}')
            break

    print(f'  최고: Epoch {best["epoch"]} | F1 {best["f1"]:.4f} | Acc {best["acc"]:.4f}')

    # 완료된 비율의 에폭 체크포인트 삭제 (Drive 용량 절약)
    if os.path.exists(epoch_ckpt_path):
        os.remove(epoch_ckpt_path)
        print(f'  에폭 체크포인트 삭제 완료')

    del model, optimizer, scaler, train_loader, train_ds
    torch.cuda.empty_cache()

    return best

## 7. 비율별 실험 실행

> 2개 비율 순차 학습. 로컬 RTX 4060 Ti 기준 약 1~2시간 소요 예상.

In [11]:
results = {}  # ratio_name → best metrics

for ratio_name, (syn_r, orig_r) in RATIOS.items():
    # 이미 완료된 비율은 건너뜀 (에폭 체크포인트 없고 모델 파일 있으면 완료로 간주)
    if os.path.exists(SAVE_PATHS[ratio_name]) and not os.path.exists(EPOCH_CKPT_PATHS[ratio_name]):
        ckpt = torch.load(SAVE_PATHS[ratio_name], map_location='cpu', weights_only=False)
        best = ckpt['best_metrics']
        results[ratio_name] = best
        print(f'[{ratio_name}] 이미 완료 (F1 {best["f1"]:.4f}), 건너뜀')
        continue

    train_df = mix_datasets(s2_train_orig, s2_synthetic,
                            syn_ratio=syn_r, orig_ratio=orig_r)
    best = train_one_ratio(
        ratio_name      = ratio_name,
        train_df        = train_df,
        save_path       = SAVE_PATHS[ratio_name],
        data_version    = f'synthetic_v2_{ratio_name}',
        epoch_ckpt_path = EPOCH_CKPT_PATHS[ratio_name],
    )
    results[ratio_name] = best

print('\n\n모든 실험 완료')


실험: ratio_5to1  |  Train 43,573건
토크나이징 중...
  배치 수: 2724
모델 초기화...
  ✅ 분류기 warm-start: stage2_model_v3.pt
  파라미터: 25.5M 학습 / 568.1M 전체


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


학습 시작 (에폭 1 ~ 30)


KeyboardInterrupt: 

## 8. 결과 비교

In [1]:
BASELINES = {
    'Stage 2 v3 (기준)':        0.4754,
    'FineTune v4 (현재 운영)':   0.4839,
    'E2E v1 (레이어 2, 원본만)': 0.4835,
    'StyleTransfer v2':         0.4690,
    'Synthetic v1 (1:3)':       0.4770,
    'Synthetic v1 (1:1)':       0.4826,
    'Synthetic v1 (3:1)':       0.4835,
}

print('=' * 65)
print('📊 전체 성능 비교')
print('=' * 65)
print(f'  {"모델":<36} {"F1":>8}  {"v3 대비":>8}')
print('-' * 65)

for name, f1 in BASELINES.items():
    diff = f1 - 0.4754
    mark = f'{diff:+.4f}' if name != 'Stage 2 v3 (기준)' else '  —'
    print(f'  {name:<36} {f1:>8.4f}  {mark:>8}')

print('-' * 65)

ratio_labels = {
    'ratio_5to1': 'Synthetic v2 (합성:원본 = 5:1)',
    'ratio_7to1': 'Synthetic v2 (합성:원본 = 7:1)',
}

best_ratio, best_f1 = None, 0.0
for ratio_name, best in results.items():
    f1   = best['f1']
    diff = f1 - 0.4754
    mark = f'{diff:+.4f}'
    label = ratio_labels[ratio_name]
    print(f'  {label:<36} {f1:>8.4f}  {mark:>8}')
    if f1 > best_f1:
        best_f1, best_ratio = f1, ratio_name

print('=' * 65)

if best_ratio:
    print(f'최고 비율: {best_ratio} (F1 = {best_f1:.4f})')
    baseline_best = max(BASELINES.values())
    if best_f1 > baseline_best:
        print(f'✅ 기존 최고(F1={baseline_best:.4f}) 대비 개선 → {SAVE_PATHS[best_ratio]}')
    elif best_f1 > 0.4754:
        print(f'↔ v3 기준 대비 향상, 기존 최고 미달')
    else:
        print(f'❌ 합성 데이터 포화 또는 오버피팅 가능성')


📊 전체 성능 비교
  모델                                         F1     v3 대비
-----------------------------------------------------------------
  Stage 2 v3 (기준)                        0.4754         —
  FineTune v4 (현재 운영)                    0.4839   +0.0085
  E2E v1 (레이어 2, 원본만)                    0.4835   +0.0081
  StyleTransfer v2                       0.4690   -0.0064
  Synthetic v1 (1:3)                     0.4770   +0.0016
  Synthetic v1 (1:1)                     0.4826   +0.0072
  Synthetic v1 (3:1)                     0.4835   +0.0081
-----------------------------------------------------------------


NameError: name 'results' is not defined

## 9. 최고 모델 상세 분석

In [12]:
# best_ratio가 결정된 후 실행
if best_ratio is None:
    raise RuntimeError('실험 결과 없음')

print(f'최고 모델 로드: {SAVE_PATHS[best_ratio]}')
ckpt      = torch.load(SAVE_PATHS[best_ratio], map_location='cpu', weights_only=False)
best_model = build_model(warmstart_path='')   # 구조만 생성
best_model.load_state_dict(ckpt['model_state_dict'])
best_model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for input_ids, attention_mask, lbl in val_loader:
        with autocast():
            logits = best_model(input_ids.to(device), attention_mask.to(device))
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(lbl.tolist())

print('─── Classification Report ───')
print(classification_report(
    all_labels, all_preds,
    target_names=list(STAGE2_CATEGORIES.values()), digits=4
))

최고 모델 로드: D:/Programming/Projects/Burnout/llm/models/stage2_model_syn_3to1.pt
  ⚠️ warm-start 모델 없음 → random init
  파라미터: 25.5M 학습 / 568.1M 전체
─── Classification Report ───
              precision    recall  f1-score   support

      정서적_고갈     0.4665    0.4893    0.4776      1167
       좌절_압박     0.5855    0.4296    0.4956      1108
    부정적_대인관계     0.4694    0.5219    0.4943      1071
        자기비하     0.4430    0.4929    0.4666      1049

    accuracy                         0.4830      4395
   macro avg     0.4911    0.4834    0.4835      4395
weighted avg     0.4916    0.4830    0.4836      4395



## 10. 직접 문장 테스트

In [13]:
TEST_SENTENCES = [
    # 일기체 (합성 데이터 스타일)
    ('오늘도 아무것도 못 한 것 같다. 그냥 지쳤다.',        '정서적_고갈'),
    ('내가 왜 이렇게 무기력한지 모르겠다.',                '정서적_고갈'),
    ('팀장이 또 내 의견을 무시했다. 진짜 억울했다.',       '좌절_압박'),
    ('오늘 회의에서 혼자 욕먹었다. 너무 불공평했다.',      '좌절_압박'),
    ('친구들이 나 빼고 놀러 갔다. 서운했다.',              '부정적_대인관계'),
    ('아무도 나를 이해 못 하는 것 같다.',                  '부정적_대인관계'),
    ('나는 왜 이렇게 못하는 걸까. 자꾸 실수만 했다.',     '자기비하'),
    ('오늘도 후회만 남았다. 내가 너무 한심하게 느껴졌다.', '자기비하'),
    # 구어체 (원본 스타일, 일기체 도메인 적응 여부 확인용)
    ('너무 지쳐서 아무것도 하기 싫어요.',                  '정서적_고갈'),
    ('팀장님이 계속 뭐라 하셔서 스트레스 받아요.',         '좌절_압박'),
]

best_model.eval()
print(f'[{best_ratio} 모델]\n')
print(f'  {"문장":<38} {"예측":<14} {"정답":<14} {"신뢰도":<8} OK?')
print('-' * 82)

with torch.no_grad():
    for text, gt in TEST_SENTENCES:
        enc = tokenizer(
            [text], padding='max_length', truncation=True,
            max_length=MAX_LEN, return_tensors='pt'
        )
        with autocast():
            logits = best_model(enc['input_ids'].to(device), enc['attention_mask'].to(device))
        probs    = torch.softmax(logits, dim=1)[0]
        pred_idx = probs.argmax().item()
        pred_cat = STAGE2_CATEGORIES[pred_idx]
        conf     = probs[pred_idx].item()
        ok       = '✅' if pred_cat == gt else '❌'
        print(f'  {text:<38} {pred_cat:<14} {gt:<14} {conf:.1%:<8} {ok}')

[ratio_3to1 모델]

  문장                                     예측             정답             신뢰도      OK?
----------------------------------------------------------------------------------


ValueError: Invalid format specifier '.1%:<8' for object of type 'float'